In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import re

C:\Users\AKASH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
threshold=[(0.5693727),(0.2735182),(0.41990516),(0.3229134),(0.2919376),(0.17040099),(0.027170286),(0.17893796)]

In [ ]:
final_threshold = [
    0.6251,   # Politics
    0.7554,   # Science and Technology
    0.4783,   # Business and Economy
    0.7679,   # International Affairs
    0.5961,   # Education
    0.7345,   # Health
    0.6283,   # Sports
    0.7477,   # Entertainment
]

In [ ]:
#final_threshold=[(0.55432695),(0.5015201),(0.21850389),(0.19308142),(0.19222741),(0.022489678),(0.014320114),(0.23062292)]

In [ ]:
category_names = [
    'Politics', 'Science and Technology', 'Business and Economy',
    'International Affairs', 'Education', 'Health', 'Sports', 'Entertainment'
]
num_labels = len(category_names)

In [6]:
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove special characters (keeping basic punctuation for BERT context)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:-]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
loaded_final_tokenizer = AutoTokenizer.from_pretrained('improved_model/best_tokenizer')
loaded_final_model = AutoModelForSequenceClassification.from_pretrained('improved_model/best_model')
loaded_final_model.to(device)
loaded_final_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [8]:
def predict_custom_news(raw_title,raw_text, model, tokenizer, device, threshold=0.5):
    
    cleaned_title = clean_text(raw_title)
    cleaned_text = clean_text(raw_text)
    threshold=torch.tensor(threshold).to(device)
    text=cleaned_title+' - '+cleaned_text

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask).logits
        probs = torch.sigmoid(logits)
        predictions_array = (probs > threshold).int().cpu().numpy()[0]
        raw_confidence = probs.cpu().numpy()[0]

    predicted_labels = []
    for i, is_predicted in enumerate(predictions_array):
        if is_predicted == 1:
            cat_name = category_names[i] if i < len(category_names) else f"Category_{i}"
            predicted_labels.append(cat_name)

    return predicted_labels, raw_confidence

In [9]:
title=input("Enter the title of the article: ")
custom_article = input("Enter the content of the article: ")
#custom_article = "Google DeepMind and OpenAI recently published new peer reviewed research regarding artificial intelligence scalability algorithms and quantum computing efficiency."
labels, scores = predict_custom_news(title,custom_article, loaded_final_model,loaded_final_tokenizer,device)

print(f"\nArticle: '{custom_article}'")
print(f"Predicted Categories: {labels}")
print(f"Raw Confidence Output: {np.round(scores, 3)}")
print(f"threshold for the categories: {threshold}")


Article: 'Google DeepMind and OpenAI recently published new peer reviewed research regarding artificial intelligence scalability algorithms and quantum computing efficiency.'
Predicted Categories: ['Technology', 'Business']
Raw Confidence Output: [0.148 0.995 0.6   0.105 0.002 0.001 0.012 0.001]
threshold for the categories: [0.5693727, 0.2735182, 0.41990516, 0.3229134, 0.2919376, 0.17040099, 0.027170286, 0.17893796]


In [10]:
category_names = [
        'Politics', 'Technology', 'Business', 'Economy',
        'International_Affairs', 'Education', 'Health', 'Science/Other'
    ]
num_labels = len(category_names)

## Explainable AI 

In [ ]:
from transformers_interpret import SequenceClassificationExplainer


In [26]:
# 1. Provide the model with the label configuration it needs
if len(loaded_final_model.config.id2label) < len(category_names):
    loaded_final_model.config.id2label = {i: name for i, name in enumerate(category_names)}
    loaded_final_model.config.label2id = {name: i for i, name in enumerate(category_names)}

In [27]:
# 2. Instantiate the Sequence Explainer
explainer = SequenceClassificationExplainer(
    loaded_final_model,
    loaded_final_tokenizer
)

In [28]:
def explain_predictions(title, text, predicted_labels):
    print("Generating explanations for the predicted categories...")
    
    cleaned_title = clean_text(title)
    cleaned_text = clean_text(text)
    combined_text = cleaned_title + ' - ' + cleaned_text

    if not predicted_labels:
        print("No categories were predicted above the threshold.")
        return

    all_explanations_dict = {}

    for label in predicted_labels:
        print(f"\n=========================================")
        print(f"Explanation for prediction: {label}")
        print(f"=========================================")
        
        # Calculate token attributions targeted at this specific predicted class
        word_attributions = explainer(combined_text, class_name=label)
        
        # Convert list of tuples to dictionary
        word_scores_dict = dict(word_attributions)
        
        # Store in master dictionary
        all_explanations_dict[label] = word_scores_dict
        
        # Optional: Print top 10 impacted words for readability
        filtered_scores = {k: v for k, v in word_scores_dict.items() if k not in ['<s>', '</s>', '<pad>', '#s', '#/s']}
        sorted_scores = sorted(filtered_scores.items(), key=lambda x: abs(x[1]), reverse=True)
        print("Top 10 impactful keywords:")
        for word, score in sorted_scores[:15]:
            print(f"  {word:15} : {score:.4f}")
            
    print("\n\n--- Final Extracted Dictionary ---")
    print(all_explanations_dict)
    
    return all_explanations_dict

In [29]:
# Run the explainer on our most recently predicted sample
word_dict_output = explain_predictions(title, custom_article, labels)


Generating explanations for the predicted categories...

Explanation for prediction: Technology


C:\Users\AKASH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers_interpret\explainers\text\sequence_classification.py:220: UserWarning: 'Technology' is not found in self.label2id keys.Defaulting to predicted index instead.
  warnings.warn(s)


Top 10 impactful keywords:
  artificial      : -0.3688
  mind            : 0.3686
  -               : 0.3399
  intelligence    : -0.1959
  ability         : -0.1955
  algorithms      : 0.1848
  recently        : 0.1830
  reviewed        : 0.1449
  google          : 0.1053
  new             : 0.0848
  published       : -0.0793
  regarding       : -0.0679
  and             : -0.0631
                  : 0.0624
  quantum         : -0.0618

Explanation for prediction: Business


C:\Users\AKASH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\transformers_interpret\explainers\text\sequence_classification.py:220: UserWarning: 'Business' is not found in self.label2id keys.Defaulting to predicted index instead.
  warnings.warn(s)


Top 10 impactful keywords:
  artificial      : -0.3688
  mind            : 0.3686
  -               : 0.3399
  intelligence    : -0.1959
  ability         : -0.1955
  algorithms      : 0.1848
  recently        : 0.1830
  reviewed        : 0.1449
  google          : 0.1053
  new             : 0.0848
  published       : -0.0793
  regarding       : -0.0679
  and             : -0.0631
                  : 0.0624
  quantum         : -0.0618


--- Final Extracted Dictionary ---
{'Technology': {'<s>': 0.0, 'research': -0.0018412385834380984, 'regarding': -0.06792891770601273, 'a': -0.04368681460618973, 'i': -0.009431789629161358, '-': 0.3398832082748413, 'google': 0.10529531538486481, 'deep': -0.032695408910512924, 'mind': 0.368622362613678, 'and': -0.06305710971355438, 'open': 0.06183614209294319, 'ai': -0.040070001035928726, 'recently': 0.18301917612552643, 'published': -0.07925068587064743, 'new': 0.08478478342294693, 'peer': 0.049368906766176224, 'reviewed': 0.14491964876651764, 'artificia